<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# Softmax (naive → online/flash)

Softmax is the first kernel in this course where *how* you reduce matters as much as *what* you
compute. A correct GPU softmax has to dodge two traps at once: floating-point **overflow**, and
**wasted passes** over the row.

We build it twice. First a straightforward **naive** kernel — three passes over the row, with a
classic shared-memory **tree reduction** for the max and the sum. It is easy to read and a good
mental model, but it touches the data more than it needs to. Then we build the **online (flash)**
kernel that fuses the max and the sum into a *single* pass, and reduces across the block with
warp shuffles plus a tiny shared-memory hand-off — the same two-level reduction pattern you will
reuse in later notebooks.

**You'll learn:** why numerically-stable softmax subtracts the row max; a shared-memory **tree
reduction** (the naive baseline); the online `(max, sum)` recurrence and why merging two partial
states *rescales* one onto the other -- captured once in a `merge()` helper; and a **two-level
reduction** (warp shuffle then shared memory).

**Runs on:** any CUDA GPU. **Prereq:** the `01_array_concepts` and `02_vector_concepts` notebooks.

In [ ]:
import cutlass
import cutlass.cute as cute
import torch

## 1. Stable softmax needs the row max

The definition is `softmax(x)_i = exp(x_i) / sum_j exp(x_j)`. Computed literally, `exp(x_i)`
overflows float32 the moment any `x_i` is even moderately large — `exp(89)` is already past the
float32 ceiling. The fix is the **numerically-stable** form, which subtracts the row max
`m = max_j x_j` before exponentiating:

```text
softmax(x)_i = exp(x_i - m) / sum_j exp(x_j - m)
```

This is the *same* math — the `exp(-m)` factor cancels between numerator and denominator — but
every exponent is now `<= 0`, so each `exp(...)` lands in `(0, 1]` and can never overflow. The
price is that, done naively, it reads the row **three times**: once to find `m`, once to sum
`exp(x - m)`, and once to normalize.

## 2. Two ways to reduce the row

| | naive (this section) | online / flash (section 5) |
|---|---|---|
| passes over the row | **3** — max, then exp, then sum | **1** — `(max, sum)` fused |
| reduction | shared-memory **tree** | warp **shuffle** + a small shared hand-off |
| extra global traffic | the `exp`s round-trip through global | none |

The naive kernel does exactly what the math says, in three passes; a shared-memory tree reduction
collapses the per-thread maxes (then sums) to one value. It's easy to read but touches the row three
times. Section 5 fuses all three into a single pass.

In [ ]:
@cute.kernel
def softmax_naive(inp: cutlass.Array, out: cutlass.Array, C: cutlass.Constexpr, BLOCK_SIZE: cutlass.Constexpr):
    """Three-pass softmax over one row, using shared-memory tree reductions."""
    shared = cutlass.Array(cutlass.Float32, BLOCK_SIZE, space=cutlass.AddressSpace.smem)
    row, _, _ = cute.arch.block_idx()   # one block per row
    tid, _, _ = cute.arch.thread_idx()
    base = row * C                      # flat (N, C) addressing: row * C + col

    # Step 1. Each thread maxes over its strided slice of the row.
    maxval = -3.4028235e38              # -FLT_MAX, the identity for max
    for i in cutlass.range_constexpr(0, C, BLOCK_SIZE):
        col = i + tid
        if col < C:
            maxval = cute.math.max(maxval, inp[base + col])

    # Step 2. Tree-reduce the per-thread maxes down to shared[0].
    shared[tid] = maxval
    for stride in [64, 32, 16, 8, 4, 2, 1]:
        cute.arch.barrier()
        if stride < BLOCK_SIZE and tid < stride:
            shared[tid] = cute.math.max(shared[tid], shared[tid + stride])
    cute.arch.barrier()
    row_max = shared[0]

    # Step 3. Write the stable exponentials exp(x - row_max).
    for i in cutlass.range_constexpr(0, C, BLOCK_SIZE):
        col = i + tid
        if col < C:
            out[base + col] = cute.math.exp(inp[base + col] - row_max, fastmath=True)
    cute.arch.barrier()

    # Step 4. Each thread sums its strided slice of the exps it just wrote.
    sumval = 0.0
    for i in cutlass.range_constexpr(0, C, BLOCK_SIZE):
        col = i + tid
        if col < C:
            sumval = sumval + out[base + col]

    # Step 5. Tree-reduce the per-thread sums down to shared[0].
    shared[tid] = sumval
    for stride in [64, 32, 16, 8, 4, 2, 1]:
        cute.arch.barrier()
        if stride < BLOCK_SIZE and tid < stride:
            shared[tid] = shared[tid] + shared[tid + stride]
    cute.arch.barrier()
    row_sum = shared[0]

    # Step 6. Normalize each exp by the row sum.
    for i in cutlass.range_constexpr(0, C, BLOCK_SIZE):
        col = i + tid
        if col < C:
            out[base + col] = out[base + col] / row_sum

## 3. Launch the naive kernel

The host entry point just launches the kernel: one block per row (`grid = (N, 1, 1)`) with
`BLOCK_SIZE` threads each. Because `C` is a `Constexpr`, the strided `range_constexpr` loops have
a compile-time trip count and are unrolled.

In [ ]:
@cute.jit
def softmax_naive_host(
    inp: cutlass.Array,
    out: cutlass.Array,
    N: cutlass.Int32,
    C: cutlass.Constexpr,
    BLOCK_SIZE: cutlass.Constexpr,
):
    softmax_naive(inp, out, C, BLOCK_SIZE).launch(
        grid=(N, 1, 1), block=(BLOCK_SIZE, 1, 1)
    )

## 4. Run and check the naive kernel

`cutlass.Array` kernel parameters accept PyTorch CUDA tensors **directly** through
`cute.runtime.from_dlpack` — no manual copy or pointer wrangling. We check the result against
`torch.softmax`.

In [ ]:
N, C = 1024, 2048
inp = torch.randn(N, C, dtype=torch.float32, device="cuda")
out = torch.zeros_like(inp)

softmax_naive_host(
    cute.runtime.from_dlpack(inp),
    cute.runtime.from_dlpack(out),
    N,
    C=C,
    BLOCK_SIZE=128,
)

torch.testing.assert_close(
    out.cpu(), torch.softmax(inp.cpu(), dim=-1), atol=1e-3, rtol=1e-3
)
print("PASS Naive")

## 5. The online (flash) trick: one pass for (max, sum)

The naive kernel needs a separate pass for the max because the stable sum `sum exp(x - m)`
depends on `m`. The **online** algorithm breaks that dependency: it computes the max and the sum
*together* in one pass by carrying a running pair `(m, s)` and **rescaling** the sum whenever the
max grows (Milakov & Gimelshein, *Online normalizer calculation for softmax*, arXiv:1805.02867).
For each new value `x`:

```text
m_new = max(m, x)
s_new = s * exp(m - m_new) + exp(x - m_new)
```

The `s * exp(m - m_new)` factor is the whole idea: every term already in `s` was measured against
the *old* max, so when the max moves we rescale the accumulated sum onto the new reference.

The same rescale lets us **merge two partial states** `(m_a, s_a)` and `(m_b, s_b)` that were
computed over disjoint chunks of the row — and it is why the merge is a rescale, **never a bare
add**:

```text
m = max(m_a, m_b)
s = s_a * exp(m_a - m) + s_b * exp(m_b - m)
```

**This kernel: one block per row, reduced in two levels.** Each thread folds its *strided* slice
of the row into a private `(m, s)` (one online pass). Threads then merge within a warp using
`shuffle_sync_down` (offsets 16, 8, 4, 2, 1 — no shared memory, no barrier), and finally across
warps through a small shared-memory array guarded by a CTA barrier. Once the block agrees on the
row's `(m, s)`, every thread writes `exp(x - m) / s`.

```text
 ONE BLOCK PER ROW   each thread t: online-fold a strided slice -> (m_t, s_t)
 STEP 1  warp reduce via shuffle_sync_down, offsets 16,8,4,2,1   (no smem, no barrier)
         warp0 -> lane0:(M0,S0)   warp1 -> lane0:(M1,S1)   ...      merge = max + exp-rescale
 STEP 2  smem[warp_id] = (M_w, S_w);  barrier;  thread0 merges all warps -> (M,S);  barrier
 FINAL   every thread writes  out_i = exp(x_i - M) / S
```

In [ ]:
def merge(m, s, other_m, other_s):
    """Merge two online-softmax (max, sum) states onto a common max -- never a bare add.

    new_max = max(m, other_m), then BOTH sums are rescaled onto it:
        new_sum = s*exp(m - new_max) + other_s*exp(other_m - new_max)
    The exp(... - new_max) factors are the rescale. Folding a single value x is just the
    same merge against the one-element state (x, 1): merge(m, s, x, 1.0).
    """
    new_max = cute.math.max(m, other_m)
    new_sum = (s * cute.math.exp(m - new_max, fastmath=True)
               + other_s * cute.math.exp(other_m - new_max, fastmath=True))
    return new_max, new_sum

In [ ]:
@cute.kernel
def softmax_kernel(inp: cutlass.Array, out: cutlass.Array, C: cutlass.Constexpr, BLOCK_SIZE: cutlass.Constexpr):
    """Row-wise numerically-stable softmax in one online (max, sum) pass.

    Inputs:
        inp  -- Float32 array viewing a flat (N, C) row-major buffer; this block reads row
                block_idx.x, the C contiguous elements starting at base = row * C.
        out  -- Float32 array over the same (N, C) buffer; written in place (same shape/dtype).
        C    -- row width (Constexpr), so the strided range_constexpr loops unroll.
        BLOCK_SIZE -- threads per block (Constexpr); a multiple of the 32-lane warp size.

    Output: writes this block's row of `out` with softmax(x)_i = exp(x_i - row_max) / row_sum,
    where row_max = max_j x_j and row_sum = sum_j exp(x_j - row_max). Each thread folds its
    strided slice into a private (max, sum) via merge(); the block reduces those partials in
    two levels (warp shuffle, then a shared-memory hand-off), and every thread normalizes.
    """
    WARP_SIZE = 32
    WARPS_PER_BLOCK = BLOCK_SIZE // WARP_SIZE
    NEG_INF = -3.4028235e38  # -FLT_MAX, the running-max identity

    # One (max, sum) partial per warp, handed off through shared memory.
    smax = cutlass.Array(cutlass.Float32, WARPS_PER_BLOCK, space=cutlass.AddressSpace.smem)
    ssum = cutlass.Array(cutlass.Float32, WARPS_PER_BLOCK, space=cutlass.AddressSpace.smem)

    row, _, _ = cute.arch.block_idx()   # one block per row
    tid, _, _ = cute.arch.thread_idx()
    warp_id = tid // WARP_SIZE
    lane_id = tid % WARP_SIZE
    base = row * C

    # Step 1. Fold this thread's strided slice into a running (max, sum), one value at a time.
    m, s = NEG_INF, 0.0
    for i in cutlass.range_constexpr(0, C, BLOCK_SIZE):
        col = i + tid
        if col < C:
            m, s = merge(m, s, inp[base + col], 1.0)

    # Step 2. Reduce across the warp with shuffles -- lane 0 ends with the warp's partial.
    for offset in [16, 8, 4, 2, 1]:
        m, s = merge(m, s, cute.arch.shuffle_sync_down(m, offset), cute.arch.shuffle_sync_down(s, offset))

    # Step 3. Each warp publishes its partial; the barrier makes them all visible.
    if lane_id == 0:
        smax[warp_id] = m
        ssum[warp_id] = s
    cute.arch.barrier()

    # Step 4. Thread 0 merges the per-warp partials into the row (max, sum).
    if tid == 0:
        m, s = smax[0], ssum[0]
        for w in cutlass.range_constexpr(1, WARPS_PER_BLOCK):
            m, s = merge(m, s, smax[w], ssum[w])
        smax[0], ssum[0] = m, s
    cute.arch.barrier()
    row_max, row_sum = smax[0], ssum[0]

    # Step 5. Write the normalized softmax.
    for i in cutlass.range_constexpr(0, C, BLOCK_SIZE):
        col = i + tid
        if col < C:
            out[base + col] = cute.math.exp(inp[base + col] - row_max, fastmath=True) / row_sum

## 6. Launch: one block per row

Same launch shape as before — grid `(N, 1, 1)` is one block per row — but with `BLOCK_SIZE = 256`
threads, i.e. 8 warps per block. As before, `C` is a `Constexpr`, so the strided
`range_constexpr` loops are unrolled.

In [ ]:
@cute.jit
def softmax(
    inp: cutlass.Array,
    out: cutlass.Array,
    N: cutlass.Int32,
    C: cutlass.Constexpr,
    BLOCK_SIZE: cutlass.Constexpr,
):
    softmax_kernel(inp, out, C, BLOCK_SIZE).launch(
        grid=(N, 1, 1), block=(BLOCK_SIZE, 1, 1)
    )

## 7. Run and check against PyTorch

Same setup as the naive run: pass the CUDA tensors straight through `cute.runtime.from_dlpack`
and compare against `torch.softmax`. The online kernel is a single fused pass plus the two-level
reduction, yet produces the identical result.

In [ ]:
N, C = 1024, 2048
inp = torch.randn(N, C, dtype=torch.float32, device="cuda")
out = torch.zeros_like(inp)

softmax(
    cute.runtime.from_dlpack(inp),
    cute.runtime.from_dlpack(out),
    N,
    C=C,
    BLOCK_SIZE=256,
)

torch.testing.assert_close(
    out.cpu(), torch.softmax(inp.cpu(), dim=-1), atol=1e-3, rtol=1e-3
)
print("PASS Optimized")

## Try it yourself

1. Change `C` (e.g. to 4096) and rerun -- the `range_constexpr` trip count changes. Does it still
   match torch?
2. **Why the merge rescales.** Build a 2-element example where merging two `(max, sum)` states by
   *adding* the sums (no `exp(Δmax)` rescale) gives the wrong answer -- the concrete reason `merge()`
   rescales instead of adding.
3. **One merge, three places.** `merge()` is called in the per-thread fold, the warp-shuffle reduce,
   and the cross-warp reduce. Trace one row's `(max, sum)` through all three and confirm the result
   is the row's true max and `sum(exp(x - max))`.